# Course 7: Privacy-Preserving Machine Learning
## Session 12: Capstone: privacy-preserving medical AI

**Goal:** write a threat model, choose among DP, federated learning and encryption, and report a measured privacy-utility trade-off.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Threat model (TODO: fill this in before any code)

| Question | Your answer |
|---|---|
| What is the data, and who are the people in it? | |
| Who holds the data? One hospital or several? | |
| Who is the adversary? (the public, the server, another hospital, an insider) | |
| What can the adversary see? (the model, its predictions, gradients, the raw data) | |
| What must stay hidden? (membership, an attribute, a record, a link) | |
| What is the privacy unit? (a record, a patient, a hospital) | |
| What accuracy is needed for the model to be worth using? | |

### 2. Choosing a tool
- The released model or its outputs could leak training data: differential privacy.
- The data cannot leave several sites: federated learning. Add secure aggregation if the server is not trusted, and DP if the final model is released.
- The client's query must be hidden from the server at prediction time: homomorphic encryption.
- A person can ask to be removed later: plan for unlearning (for example shards) from the start.

These answer different threats, so real systems combine them.

In [ ]:
def suggest(model_released, data_in_several_sites, server_trusted, hide_queries, deletion_requests):
    tools = []
    if model_released: tools.append("DP training (DP-SGD), with epsilon and delta reported")
    if data_in_several_sites:
        tools.append("federated learning")
        if not server_trusted: tools.append("secure aggregation")
    if hide_queries: tools.append("encrypted inference (CKKS), with a shallow polynomial model")
    if deletion_requests: tools.append("sharded training so that unlearning is cheap")
    return tools or ["no privacy tool needed for this threat model; say why"]

print(suggest(model_released=True, data_in_several_sites=True, server_trusted=False, hide_queries=False, deletion_requests=True))

### 3. A measurement harness
One function that trains at a given epsilon and reports utility and an attack result together. Here it runs on the breast cancer dataset shipped with scikit-learn. Replace the data with your own.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from math import comb, exp, log, sqrt
from torch.func import functional_call, vmap, grad

# ---- privacy accounting (Renyi DP of the sampled Gaussian mechanism, whole-number orders) ----
def rdp(q, sigma, alpha):
    """Renyi DP at order alpha of one DP-SGD step. q = sampling rate, sigma = noise multiplier."""
    if q == 1.0: return alpha / (2 * sigma ** 2)
    terms = [log(comb(alpha, k)) + (alpha - k) * log(1 - q) + k * log(q) + (k * k - k) / (2 * sigma ** 2) for k in range(alpha + 1)]
    m = max(terms)
    return (m + log(sum(exp(t - m) for t in terms))) / (alpha - 1)

def epsilon(q, sigma, steps, delta):
    """Convert composed Renyi DP to (epsilon, delta)-DP and keep the best order."""
    return min(steps * rdp(q, sigma, a) + log(1 / delta) / (a - 1) for a in range(2, 129))

# ---- DP-SGD ----
def dp_sgd(model, X, y, sigma, clip=1.0, lr=0.5, steps=300, q=0.1, seed=0, loss_fn=F.cross_entropy):
    """Poisson sampling, per-example clipping, Gaussian noise. sigma=0 with clip=inf is ordinary SGD."""
    g = torch.Generator().manual_seed(seed)
    params = {k: v.detach().clone() for k, v in model.named_parameters()}
    one = lambda p, x, t: loss_fn(functional_call(model, p, (x[None],)), t[None])
    per_example_grads = vmap(grad(one), in_dims=(None, 0, 0))
    n = len(X)
    for _ in range(steps):
        idx = torch.nonzero(torch.rand(n, generator=g) < q).squeeze(-1)
        grads = per_example_grads(params, X[idx], y[idx])
        norms = torch.sqrt(sum((v.flatten(1) ** 2).sum(1) for v in grads.values()))
        scale = (clip / (norms + 1e-12)).clamp(max=1.0)                     # shrink any gradient longer than clip
        for k in params:
            total = (grads[k] * scale.view(-1, *[1] * (grads[k].dim() - 1))).sum(0)
            if sigma > 0: total = total + torch.randn(total.shape, generator=g) * sigma * clip
            params[k] -= lr * total / (q * n)
    model.load_state_dict(params)
    return model

def mlp(d_in, n_out, hidden=32):
    return nn.Sequential(nn.Linear(d_in, hidden), nn.Tanh(), nn.Linear(hidden, n_out))

def accuracy(model, X, y):
    with torch.no_grad(): return (model(X).argmax(1) == y).float().mean().item()

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(0)
X, y = load_breast_cancer(return_X_y=True); X = StandardScaler().fit_transform(X)   # for a real project, fit the scaler privately or on public data
X = X / np.maximum(1, np.linalg.norm(X, axis=1, keepdims=True) / 6)
perm = rng.permutation(len(X)); mem, non = perm[:280], perm[280:]
y_rec = np.where(rng.random(len(y)) < 0.15, 1 - y, y)          # recorded labels: 15% are wrong, as in real records
Xt, yt = torch.tensor(X, dtype=torch.float32), torch.tensor(y_rec)
STEPS, Q, DELTA = 200, 0.2, 1e-3

def measure(sigma, seed):
    torch.manual_seed(seed)
    private = sigma is not None
    m = dp_sgd(mlp(30, 2, hidden=256), Xt[mem], yt[mem], sigma=sigma if private else 0.0, clip=1.0 if private else float("inf"),
               lr=1.0 if private else 0.5, steps=STEPS if private else 2000, q=Q, seed=seed)
    with torch.no_grad():
        out = m(Xt)
        utility = roc_auc_score(y[non], out[non, 1] - out[non, 0])          # scored against the true diagnosis
        loss = F.cross_entropy(out, yt, reduction="none").numpy()
    attack = roc_auc_score(np.r_[np.ones(len(mem)), np.zeros(len(non))], -np.r_[loss[mem], loss[non]])
    return utility, attack

print("epsilon   test AUC (mean, sd)   membership attack AUC")
table = []
for sigma in [None, 0.7, 1.5, 4.0]:
    r = np.array([measure(sigma, s) for s in range(3)])
    eps = float("inf") if sigma is None else epsilon(Q, sigma, STEPS, DELTA)
    table.append((eps, r[:, 0].mean(), r[:, 1].mean()))
    print(f"{eps:7.2f}   {r[:, 0].mean():.3f} ± {r[:, 0].std():.3f}       {r[:, 1].mean():.3f}")
assert table[0][2] > table[-1][2] + 0.05

The non-private model is trained for a long time on purpose, so that it memorizes the wrong labels and the attack has something to find. With DP the attack falls to about 0.5, and at a large epsilon the test AUC is even better than without privacy, because clipping and noise stop the model memorizing label errors. At small epsilon the utility drops. Your capstone needs this kind of table for your own task.

### 4. What the report must contain (TODO)
1. The threat model table, filled in.
2. The tool or tools chosen, and one that was rejected, with reasons.
3. The privacy claim in one sentence: epsilon, delta, the privacy unit, and what was not protected (for example, hyperparameter tuning).
4. A privacy-utility table with at least 3 seeds per row and a non-private baseline.
5. One attack run against your own system (membership inference or a canary audit).
6. Subgroup results: who pays the accuracy cost?
7. Limits: what would break the guarantee in deployment?

### 5. Presentation (10 minutes)
- 1 slide: the setting and the threat model.
- 1 slide: the design.
- 2 slides: results, including the attack.
- 1 slide: limits and what you would do with three more months.